# encoder-decoder-symmetric — faded example 3: Encoder-Decoder Channel Trace: Build from Config

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `encoder-decoder-symmetric`. The last cell reports your progress on the `CNN: Encoder-decoder symmetric layout` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: Encoder-decoder symmetric layout` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`encoder-decoder-symmetric`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "encoder-decoder-symmetric"
DD_SUBTOPIC = "CNN: Encoder-decoder symmetric layout"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Given a list of encoder channel sizes and a matching spatial downsampling factor, the symmetric decoder's channel list is simply the encoder list reversed. If the encoder channel list is `[C_in, 16, 32, 64]`, the decoder takes `[64, 32, 16, C_in]`. Building the module from these mirror lists guarantees structural symmetry without hand-counting.

## Faded exercise 3

Implement `build_ae_from_channels(channel_list, pool_factor=2)` where `channel_list` is the encoder's channel sequence (e.g. `[1, 8, 16]`). Build an encoder with `len(channel_list)-1` stages (each Conv2d+ReLU+MaxPool2d(pool_factor)), then build a decoder that mirrors it using Upsample+Conv2d. Return an `nn.Module`.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t
import torch.nn as nn

def build_ae_from_channels(channel_list, pool_factor: int = 2) -> nn.Module:
    class AE(nn.Module):
        def __init__(self):
            super().__init__()
            enc_layers = []
            for in_c, out_c in zip(channel_list[:-1], channel_list[1:]):
                enc_layers += [nn.Conv2d(in_c, out_c, 3, padding=1), nn.ReLU(), nn.MaxPool2d(pool_factor)]
            self.encoder = nn.Sequential(*enc_layers)
            rev = list(reversed(channel_list))
            dec_layers = []
            for in_c, out_c in zip(rev[:-1], rev[1:]):
                dec_layers += [nn.Upsample(scale_factor=pool_factor), nn.Conv2d(in_c, out_c, 3, padding=1)]
                if out_c != channel_list[0]:
                    dec_layers.append(nn.ReLU())
            self.decoder = nn.Sequential(*dec_layers)

        def forward(self, x):
            return self.decoder(self.encoder(x))
    return AE()


def _test():
    import torch as t
    model = build_ae_from_channels([1, 8, 16], pool_factor=2)
    model.eval()
    x = t.randn(2, 1, 16, 16)
    out = model(x)
    assert out.shape == x.shape, f"Expected (2,1,16,16), got {out.shape}"
    model2 = build_ae_from_channels([3, 16, 32, 64], pool_factor=2)
    x2 = t.randn(2, 3, 32, 32)
    out2 = model2(x2)
    assert out2.shape == x2.shape, f"Expected (2,3,32,32), got {out2.shape}"


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

def build_ae_from_channels(channel_list, pool_factor: int = 2) -> nn.Module:
    class AE(nn.Module):
        def __init__(self):
            super().__init__()
            enc_layers = []
            for in_c, out_c in zip(channel_list[:-1], channel_list[1:]):
                enc_layers += [nn.Conv2d(in_c, out_c, 3, padding=1), nn.ReLU(), nn.MaxPool2d(pool_factor)]
            self.encoder = nn.Sequential(*enc_layers)
            rev = list(reversed(channel_list))
            dec_layers = []
            for in_c, out_c in zip(rev[:-1], rev[1:]):
                dec_layers += [nn.Upsample(scale_factor=pool_factor), nn.Conv2d(in_c, out_c, 3, padding=1)]
                if out_c != channel_list[0]:
                    dec_layers.append(nn.ReLU())
            self.decoder = nn.Sequential(*dec_layers)

        def forward(self, x):
            return self.decoder(self.encoder(x))
    return AE()
```
</details>